# NB05 — Train PPO (Apple Full-Body) — RTX 5090 Optimized

Train a **PPO** agent on `UnitreeG1PlaceAppleInBowlFullBody-v1` using
Stable-Baselines3. Optimized for **RTX 5090 (32 GB VRAM, 40 GB RAM)**.
Same training budget as SAC (NB06) for fair comparison.

| Parameter | Value |
|-----------|-------|
| Algorithm | PPO (Proximal Policy Optimization) |
| Library | Stable-Baselines3 |
| Policy | MlpPolicy [512, 512] **ReLU** (~790K params) |
| Budget | **2,000,000** env steps (GPU) / 20K (CPU demo) |
| Envs | **64** GPU-vectorized parallel environments |
| LR | **3e-4 → 1e-5** linear decay |
| Robot | Unitree G1 Full Body (37 DOF) |


## Objectives

1. Create env with VecNormalize + proper wrappers.
2. Configure SB3 PPO with RTX 5090-optimized hyperparameters.
3. Train for `TOTAL_ENV_STEPS` (2M) with checkpointing every 200K.
4. Save model + learning curve.
5. Quick evaluation (20 deterministic episodes).


## Resources

| Resource | Requirement | Notes |
|----------|-------------|-------|
| GPU | **RTX 5090** (32 GB VRAM) | 64 vectorized envs |
| RAM | 40 GB | VecNormalize + large batch |
| Runtime | ~2-4 hours (2M on RTX 5090) | CPU: 20K demo only |


## Imports & Setup


In [ ]:
import sys, os, json, random, time
from pathlib import Path

import numpy as np
import torch
import gymnasium as gym
import matplotlib.pyplot as plt
import pandas as pd

from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import BaseCallback, CheckpointCallback
from stable_baselines3.common.vec_env import VecNormalize

import mani_skill.envs
from mani_skill.utils.wrappers.gymnasium import CPUGymWrapper

PROJECT_ROOT = Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
from src.envs import UnitreeG1PlaceAppleInBowlFullBodyEnv

ARTIFACTS_DIR = PROJECT_ROOT / "artifacts" / "NB05"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

IS_GPU = torch.cuda.is_available()
DEVICE = "cuda" if IS_GPU else "cpu"
print(f"Device: {DEVICE}")
if IS_GPU:
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")


## Configuration


In [ ]:
CFG = {
    "seed":            42,
    "env_id":          "UnitreeG1PlaceAppleInBowlFullBody-v1",
    "control_mode":    "pd_joint_delta_pos",
    "obs_mode":        "state",
    # ── Training budget (RTX 5090 optimized) ──
    "total_env_steps": 2_000_000 if IS_GPU else 20_000,
    "n_envs":          64 if IS_GPU else 1,
    # ── PPO Hyperparameters (RTX 5090) ──
    "learning_rate":   3e-4,     # initial LR (linear decay to 1e-5)
    "lr_end":          1e-5,
    "n_steps":         2048,
    "batch_size":      2048,     # large batch for 64 envs
    "n_epochs":        10,
    "gamma":           0.99,
    "gae_lambda":      0.95,
    "clip_range":      0.2,
    "ent_coef":        0.01,
    "vf_coef":         0.5,
    "max_grad_norm":   0.5,
    "net_arch":        [512, 512],
    "activation_fn":   "ReLU",
    # ── Normalization ──
    "vec_normalize":   True,     # VecNormalize wrapper
    "norm_obs":        True,
    "norm_reward":     True,
    "clip_obs":        10.0,
    # ── Checkpointing ──
    "checkpoint_freq": 200_000,  # save every 200K steps
    # ── Eval ──
    "eval_episodes":   20,
}

SEED = CFG["seed"]
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

with open(ARTIFACTS_DIR / "nb05_config.json", "w") as f:
    json.dump(CFG, f, indent=2)

print("PPO Config (RTX 5090 optimized):")
for k, v in CFG.items():
    print(f"  {k}: {v}")


## Step 1 — Linear Learning Rate Schedule


In [ ]:
def linear_schedule(initial_lr: float, final_lr: float = 1e-5):
    """Linear LR decay: initial_lr → final_lr over training."""
    def schedule(progress_remaining: float) -> float:
        return final_lr + (initial_lr - final_lr) * progress_remaining
    return schedule

lr_schedule = linear_schedule(CFG["learning_rate"], CFG["lr_end"])
print(f"LR schedule: {CFG['learning_rate']} → {CFG['lr_end']} (linear decay)")


## Step 2 — Create Environment (64 envs + VecNormalize)


In [ ]:
def make_train_env(env_id, n_envs=1, seed=42):
    """Create SB3-compatible env with proper wrappers."""
    env = gym.make(
        env_id,
        num_envs=n_envs,
        obs_mode=CFG["obs_mode"],
        control_mode=CFG["control_mode"],
        render_mode="rgb_array",
    )
    if n_envs == 1:
        env = CPUGymWrapper(env)
    return env

train_env = make_train_env(CFG["env_id"], n_envs=CFG["n_envs"], seed=SEED)
eval_env = make_train_env(CFG["env_id"], n_envs=1, seed=9999)

# Apply VecNormalize if configured
if CFG["vec_normalize"] and hasattr(train_env, "observation_space"):
    try:
        train_env = VecNormalize(
            train_env,
            norm_obs=CFG["norm_obs"],
            norm_reward=CFG["norm_reward"],
            clip_obs=CFG["clip_obs"],
        )
        print("✅ VecNormalize applied (norm_obs + norm_reward)")
    except Exception as e:
        print(f"⚠️ VecNormalize skipped: {e}")

print(f"Train env: n_envs={CFG['n_envs']}, obs={train_env.observation_space.shape}, "
      f"act={train_env.action_space.shape}")
print(f"Eval env:  n_envs=1, obs={eval_env.observation_space.shape}")


## Step 3 — Configure PPO (RTX 5090)


In [ ]:
model = PPO(
    "MlpPolicy",
    train_env,
    learning_rate=lr_schedule,
    n_steps=CFG["n_steps"],
    batch_size=CFG["batch_size"],
    n_epochs=CFG["n_epochs"],
    gamma=CFG["gamma"],
    gae_lambda=CFG["gae_lambda"],
    clip_range=CFG["clip_range"],
    ent_coef=CFG["ent_coef"],
    vf_coef=CFG["vf_coef"],
    max_grad_norm=CFG["max_grad_norm"],
    policy_kwargs={
        "net_arch": CFG["net_arch"],
        "activation_fn": torch.nn.ReLU,
    },
    verbose=1,
    seed=SEED,
    device=DEVICE,
)

total_params = sum(p.numel() for p in model.policy.parameters())
print(f"PPO model created: {total_params:,} parameters (~790K expected)")
print(f"  net_arch: {CFG['net_arch']}, activation: ReLU")
print(f"  LR: {CFG['learning_rate']} → {CFG['lr_end']} (linear decay)")


## Step 4 — Training Callbacks


In [ ]:
class TrainLogCallback(BaseCallback):
    """Log episode rewards and lengths during training."""

    def __init__(self):
        super().__init__()
        self.episode_rewards = []
        self.episode_lengths = []

    def _on_step(self):
        infos = self.locals.get("infos", [])
        for info in (infos if isinstance(infos, list) else [infos]):
            if isinstance(info, dict) and "episode" in info:
                self.episode_rewards.append(float(info["episode"]["r"]))
                self.episode_lengths.append(int(info["episode"]["l"]))
        return True

log_cb = TrainLogCallback()

# Checkpoint callback (save every 200K steps)
ckpt_cb = CheckpointCallback(
    save_freq=max(CFG["checkpoint_freq"] // CFG["n_envs"], 1),
    save_path=str(ARTIFACTS_DIR),
    name_prefix="checkpoint",
)
print(f"✅ Callbacks ready (log + checkpoint every {CFG['checkpoint_freq']:,} steps)")


## Step 5 — Train PPO (2M steps)


In [ ]:
print(f"\n{'='*60}")
print(f"  Training PPO — {CFG['total_env_steps']:,} steps")
print(f"  n_envs={CFG['n_envs']}, batch={CFG['batch_size']}, device={DEVICE}")
print(f"  net_arch={CFG['net_arch']}, activation=ReLU")
print(f"  LR: {CFG['learning_rate']} → {CFG['lr_end']}")
print(f"{'='*60}")

start_time = time.time()
model.learn(
    total_timesteps=CFG["total_env_steps"],
    callback=[log_cb, ckpt_cb],
    progress_bar=True,
)
elapsed = time.time() - start_time
print(f"\nTraining completed in {elapsed:.1f}s "
      f"({elapsed/60:.1f} min, {CFG['total_env_steps']/elapsed:.0f} steps/s)")


## Step 6 — Save Model


In [ ]:
model.save(str(ARTIFACTS_DIR / "ppo_apple"))
print(f"✅ Model saved: {ARTIFACTS_DIR / 'ppo_apple.zip'}")

# Save VecNormalize stats if used
if CFG["vec_normalize"]:
    try:
        train_env.save(str(ARTIFACTS_DIR / "vec_normalize.pkl"))
        print("✅ VecNormalize stats saved")
    except Exception:
        pass


## Step 7 — Quick Evaluation (20 episodes)


In [ ]:
eval_results = []
for ep in range(CFG["eval_episodes"]):
    obs, info = eval_env.reset(seed=ep * 100)
    ep_reward, ep_steps = 0.0, 0
    for step in range(1000):
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = eval_env.step(action)
        ep_reward += float(reward)
        ep_steps += 1
        if terminated or truncated:
            break
    success = bool(info.get("success", False))
    eval_results.append({
        "episode": ep, "total_reward": ep_reward,
        "steps": ep_steps, "success": success,
    })
    print(f"  Ep {ep+1:2d}: reward={ep_reward:.4f}, steps={ep_steps}, success={success}")

eval_summary = {
    "mean_reward":     float(np.mean([r["total_reward"] for r in eval_results])),
    "std_reward":      float(np.std([r["total_reward"] for r in eval_results])),
    "success_rate":    float(np.mean([r["success"] for r in eval_results])),
    "mean_steps":      float(np.mean([r["steps"] for r in eval_results])),
    "training_time_s": elapsed,
    "total_steps":     CFG["total_env_steps"],
}

with open(ARTIFACTS_DIR / "eval_results.json", "w") as f:
    json.dump(eval_summary, f, indent=2)

print(f"\nPPO Eval Summary:")
for k, v in eval_summary.items():
    print(f"  {k}: {v}")


## Step 8 — Learning Curve


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ep_rewards = log_cb.episode_rewards
ep_lengths = log_cb.episode_lengths

if ep_rewards:
    axes[0].plot(ep_rewards, alpha=0.3, color="steelblue", linewidth=0.5)
    window = min(50, len(ep_rewards))
    if len(ep_rewards) >= window:
        rolling = np.convolve(ep_rewards, np.ones(window)/window, mode="valid")
        axes[0].plot(range(window-1, len(ep_rewards)), rolling,
                     color="darkblue", linewidth=2, label=f"Rolling avg ({window})")
    axes[0].set_title("Episode Rewards During Training")
    axes[0].set_xlabel("Episode")
    axes[0].set_ylabel("Total Reward")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

if ep_lengths:
    axes[1].plot(ep_lengths, alpha=0.3, color="seagreen", linewidth=0.5)
    if len(ep_lengths) >= window:
        rolling_len = np.convolve(ep_lengths, np.ones(window)/window, mode="valid")
        axes[1].plot(range(window-1, len(ep_lengths)), rolling_len,
                     color="darkgreen", linewidth=2)
    axes[1].set_title("Episode Lengths")
    axes[1].set_xlabel("Episode")
    axes[1].set_ylabel("Steps")
    axes[1].grid(True, alpha=0.3)

fig.suptitle(f"NB05 — PPO Training ({CFG['total_env_steps']:,} steps, RTX 5090)",
             fontweight="bold")
fig.tight_layout()
fig.savefig(ARTIFACTS_DIR / "learning_curve.png", dpi=150)
plt.show()

# Save training log
training_log = pd.DataFrame({
    "episode": range(len(ep_rewards)),
    "reward": ep_rewards,
    "length": ep_lengths[:len(ep_rewards)] if ep_lengths else [],
})
training_log.to_csv(ARTIFACTS_DIR / "training_log.csv", index=False)
print("Saved: learning_curve.png, training_log.csv")


## Cleanup


In [ ]:
train_env.close()
eval_env.close()
print("✅ NB05 PPO Training Complete")


## Artifacts

| File | Description |
|------|-------------|
| `ppo_apple.zip` | Trained PPO model (SB3 format) |
| `vec_normalize.pkl` | VecNormalize statistics |
| `checkpoint_*.zip` | Checkpoints every 200K steps |
| `learning_curve.png` | Episode reward over training |
| `training_log.csv` | Per-episode reward/length during training |
| `eval_results.json` | Quick eval (20 episodes) after training |
| `nb05_config.json` | Full config (hyperparameters + env config) |

## RTX 5090 Optimization Notes

- **64 vectorized envs** on GPU → high throughput
- **[512, 512] ReLU** network (~790K params) — larger capacity
- **Batch size 2048** — matches n_steps for efficient gradient updates
- **Linear LR decay** (3e-4 → 1e-5) — smooth convergence
- **VecNormalize** — stabilizes training with obs/reward normalization
- **Checkpoints** every 200K — resume if interrupted
- **Fairness**: `total_env_steps`, `net_arch`, `VecNormalize` MUST match NB06/NB07
- On CPU mode (20K steps), results are for pipeline testing only
